In [ ]:
"""
Merge Google Street View metadata with accident records.

Overview
--------
This script enriches an accident dataset with two additional features derived
from Google Street View metadata:

1. streetview_distance_m
   - The geodesic distance (in meters) between the accident location and the
     retrieved Street View panorama location.

2. streetview_time_diff_days
   - The number of days between the accident timestamp and the Street View
     imagery date.

The script matches records using the requested latitude and longitude used
during Street View retrieval.

Input Files
-----------
1. final_file_sin_extra.csv
   Accident dataset containing:
   - latitude
   - longitude
   - accident_time_utc
   - other accident-related variables

2. streetview_metadata_compiled.csv
   Street View metadata containing:
   - requested_latitude
   - requested_longitude
   - retrieved_latitude
   - retrieved_longitude
   - time_utc (stored as YYYY-MM)

Processing Steps
----------------
1. Load both datasets.
2. Convert accident timestamps to UTC datetime.
3. Convert Street View month information into a representative date
   (15th day of the month).
4. Build a lookup dictionary indexed by
      (requested_latitude, requested_longitude).
5. For every accident:
      • Find matching Street View metadata.
      • Compute the geodesic distance between the accident location and each
        retrieved panorama.
      • Select the panorama with the smallest distance.
      • Compute the time difference (days) between the accident date and
        imagery date.
      • Store both values in new columns.
6. Save the enriched dataset.

Output
------
merged_dataset_2.csv

New Columns
-----------
streetview_distance_m
    Distance (meters) from accident location to the selected Street View
    panorama.

streetview_time_diff_days
    Number of days between the accident timestamp and the Street View
    imagery date.

Dependencies
------------
- pandas
- geopy
- tqdm
"""

In [ ]:
import pandas as pd
from geopy.distance import geodesic
from tqdm.auto import tqdm

# ==========================================================
# Input / Output
# ==========================================================

ACCIDENTS_FILE = "final_file_sin_extra.csv"
METADATA_FILE = "streetview_metadata_compiled.csv"

OUTPUT_FILE = "merged_dataset_2.csv"

# ==========================================================
# Read files
# ==========================================================

print("Reading files...")

accidents = pd.read_csv(ACCIDENTS_FILE)
metadata = pd.read_csv(METADATA_FILE)

print("\n=== Accident columns ===")
print(accidents.columns.tolist())

print("\n=== Metadata columns ===")
print(metadata.columns.tolist())

print("\n=== First 5 accident keys ===")
print(accidents[["latitude", "longitude"]].head())

print("\n=== First 5 metadata keys ===")
print(metadata[["requested_latitude", "requested_longitude"]].head())

# ==========================================================
# Convert dates
# ==========================================================

# Accident timestamp
accidents["accident_time_utc"] = pd.to_datetime(
    accidents["accident_time_utc"],
    utc=True,
    errors="coerce"
)

# Google Street View only provides YYYY-MM.
# Convert to the 15th of the month.
metadata["streetview_date"] = pd.to_datetime(
    metadata["time_utc"].astype(str) + "-15",
    utc=True,
    errors="coerce"
)

# ==========================================================
# Prepare output columns
# ==========================================================

accidents["streetview_distance_m"] = pd.NA
accidents["streetview_time_diff_days"] = pd.NA

# ==========================================================
# Build lookup by requested coordinates
# ==========================================================

print("Building lookup...")

lookup = {}

for _, row in metadata.iterrows():

    key = (
        row["requested_latitude"],
        row["requested_longitude"]
    )

    lookup.setdefault(key, []).append(row)

# ==========================================================
# Update accident rows
# ==========================================================

updated_rows = 0

print("Computing features...")

for idx, accident in tqdm(
    accidents.iterrows(),
    total=len(accidents),
    desc="Processing accidents"
):

    key = (
        accident["latitude"],
        accident["longitude"]
    )

    if key not in lookup:
        continue

    matches = lookup[key]

    # If multiple metadata rows exist, choose the closest panorama
    # (smallest physical distance)
    best_distance = None
    best_days = None

    for m in matches:

        distance = geodesic(
            (accident["latitude"], accident["longitude"]),
            (m["retrieved_latitude"], m["retrieved_longitude"])
        ).meters

        if pd.notna(m["streetview_date"]) and pd.notna(accident["accident_time_utc"]):
            days = (m["streetview_date"] - accident["accident_time_utc"]).days
        else:
            days = pd.NA

        if best_distance is None or distance < best_distance:
            best_distance = distance
            best_days = days

    accidents.at[idx, "streetview_distance_m"] = best_distance
    accidents.at[idx, "streetview_time_diff_days"] = best_days

    updated_rows += 1

# ==========================================================
# Save
# ==========================================================

accidents.to_csv(OUTPUT_FILE, index=False)

print("\nDone.")
print(f"Rows updated : {updated_rows:,}")
print(f"Rows total   : {len(accidents):,}")
print(f"Output       : {OUTPUT_FILE}")